In [ ]:
# 1. importing required libraries
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt


In [ ]:
from google.colab import files

uploaded = files.upload()   # Choose your ZIP archive here

In [ ]:
import os

print(os.listdir('/content'))  # just to confirm the name

!unzip "/content/Retail Analytics & AI-Powered Sales Forecasting System" -d "/content/unzipped"

In [ ]:
import pandas as pd

# after the unzip cell
data = pd.read_csv("/content/unzipped/Retail_Sales_Data_Unlox (1).csv")
data.head()
data.columns


In [ ]:
# rename for convenience
data = data.rename(columns={'Date': 'date', 'Total_Sales': 'sales'})

# convert types
data['date'] = pd.to_datetime(data['date'])
data['sales'] = pd.to_numeric(data['sales'], errors='coerce')
data = data.dropna(subset=['sales'])

# sort by date
data = data.sort_values('date').reset_index(drop=True)


In [ ]:
# rename for convenience
data = data.rename(columns={'Date': 'date', 'Total_Sales': 'sales'})

# convert types
data['date'] = pd.to_datetime(data['date'])
data['sales'] = pd.to_numeric(data['sales'], errors='coerce')
data = data.dropna(subset=['sales'])

# sort by date
data = data.sort_values('date').reset_index(drop=True)

In [ ]:
# 1) create df from data with only date and sales
df = data[['date', 'sales']].copy()


In [ ]:
# 2) create time index and lag features
df['t'] = np.arange(1, len(df) + 1)

max_lag = 12
for k in range(1, max_lag + 1):
    df[f'lag_{k}'] = df['sales'].shift(k)

df_model = df.dropna().reset_index(drop=True)

feature_cols = [f'lag_{k}' for k in range(1, max_lag + 1)]
X = df_model[feature_cols].values
y = df_model['sales'].values

split_idx = int(len(X) * 0.8)
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]
dates_test = df_model['date'][split_idx:]

model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("Test RMSE:", rmse)
print("Test R^2 :", r2)

In [ ]:
# use df_model, y_test, y_pred, and dates_test from earlier
plt.figure(figsize=(10, 5))
plt.plot(dates_test, y_test, label='Actual sales')
plt.plot(dates_test, y_pred, label='Predicted sales', linestyle='--')
plt.xlabel('Date')
plt.ylabel('Sales')
plt.title('Actual vs Predicted Sales (Test Set)')
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# function to forecast future periods using last 12 sales
def forecast_future(model, last_sales, n_steps=6):
    history = list(last_sales)
    forecasts = []
    for _ in range(n_steps):
        x_input = np.array(history[-12:]).reshape(1, -1)
        y_hat = model.predict(x_input)[0]
        forecasts.append(y_hat)
        history.append(y_hat)
    return forecasts

last_12 = df['sales'].iloc[-12:].values
future_6 = forecast_future(model, last_12, n_steps=6)
print("Forecasted sales for next 6 periods:", future_6)

In [ ]:
freq = 'D'  # 'W' for weekly, 'M' for monthly

last_date = df['date'].iloc[-1]
future_dates = pd.date_range(start=last_date + pd.Timedelta(1, unit=freq[0]),
                             periods=6, freq=freq)

future_df = pd.DataFrame({'date': future_dates, 'forecast_sales': future_6})
future_df


In [ ]:
# coefficient magnitude as simple importance
coef_importance = pd.DataFrame(
    {'feature': feature_cols, 'coef': model.coef_}
).sort_values('coef', key=np.abs, ascending=False)
coef_importance


In [ ]:
future_df.to_csv('future_sales_forecast.csv', index=False)
from google.colab import files
files.download('future_sales_forecast.csv')

In [ ]:
residuals = y_test - y_pred

print("Residual mean:", residuals.mean())
print("Residual std:", residuals.std())

plt.figure(figsize=(8,4))
plt.hist(residuals, bins=30)
plt.title("Residual distribution")
plt.xlabel("Error")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10,5))
plt.plot(df['date'], df['sales'])
plt.xlabel('Date')
plt.ylabel('Sales')
plt.title('Sales over time')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# join back with original data to use more features
analysis_df = data.copy()

# average sales by store
store_sales = analysis_df.groupby('Store_ID')['sales'].mean().sort_values(ascending=False)
print("Average sales by store:")
print(store_sales.head(10))

# average sales by product category
cat_sales = analysis_df.groupby('Product_Category')['sales'].mean().sort_values(ascending=False)
print("\nAverage sales by product category:")
print(cat_sales.head(10))

In [ ]:
num_cols = ['sales', 'Unit_Price', 'Units_Sold', 'Discount_in_%', 'Revenue', 'Stock_On_Hand']
num_cols = [c for c in num_cols if c in analysis_df.columns]

corr = analysis_df[num_cols].corr()
print(corr['sales'].sort_values(ascending=False))

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor

# copying and cleaning relevant columns
full = data.copy()

# make sure we still have 'sales' as the target
# (we already renamed Total_Sales -> sales before)

# ensuring numeric for useful numeric features
num_cols = ['sales', 'Unit_Price', 'Units_Sold',
            'Discount_Percentage', 'Revenue', 'Stock_On_Hand']
for c in num_cols:
    if c in full.columns:
        full[c] = pd.to_numeric(full[c], errors='coerce')

full = full.dropna(subset=['sales'])

# encoding categoricals with one-hot
cat_cols = ['Store_ID', 'Store_Location', 'Product_ID', 'Product_Category',
            'Product_Subcategory', 'Brand', 'Customer_Type',
            'Payment_Mode', 'Promotion_Applied', 'Region', 'Holiday_Flag']
cat_cols = [c for c in cat_cols if c in full.columns]

full_encoded = pd.get_dummies(full, columns=cat_cols, drop_first=True)

X_full = full_encoded.drop(columns=['sales', 'date'])
y_full = full_encoded['sales']

In [ ]:
# example: picking some numeric features
feature_cols_full = ['Unit_Price', 'Units_Sold', 'Discount_Percentage', 'Stock_On_Hand', 'Revenue']
feature_cols_full = [c for c in feature_cols_full if c in data.columns]

Xf = data[feature_cols_full].values
yf = data['sales'].values

Xf_train, Xf_test, yf_train, yf_test = train_test_split(Xf, yf, test_size=0.2, random_state=42)


In [ ]:
rf = RandomForestRegressor(
    n_estimators=50,
    max_depth=10,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)
rf.fit(Xf_train, yf_train)
yf_pred = rf.predict(Xf_test)

In [ ]:
from sklearn.metrics import mean_squared_error, r2_score

mse_rf = mean_squared_error(yf_test, yf_pred)
rmse_rf = mse_rf ** 0.5    # or np.sqrt(mse_rf)
r2_rf = r2_score(yf_test, yf_pred)

In [ ]:
# 1) basic performance segments by average sales
store_perf = analysis_df.groupby('Store_ID')['sales'].mean().reset_index()
q1 = store_perf['sales'].quantile(0.33)
q2 = store_perf['sales'].quantile(0.66)

def label_segment(x):
    if x <= q1:
        return 'Low'
    elif x <= q2:
        return 'Medium'
    else:
        return 'High'

store_perf['segment'] = store_perf['sales'].apply(label_segment)
store_perf.head()


In [ ]:
store_behavior = (
    analysis_df
    .groupby('Store_ID')
    .agg({
        'sales': 'mean',
        'Discount_Percentage': 'mean',
        'Stock_On_Hand': 'mean',
        'Store_Rating': 'mean'
    })
    .reset_index()
)
store_behavior.head()

In [ ]:
!pip install -q plotly ipywidgets

import plotly.express as px
import ipywidgets as widgets
from IPython.display import display

#enabling widgets in colab
from google.colab import output
output.enable_custom_widget_manager()

In [ ]:
# mergeing performance segment and behavior features
store_view = store_behavior.merge(store_perf[['Store_ID', 'segment']],
                                  on='Store_ID', how='left')

# dropdown to choose segment
segment_dropdown = widgets.Dropdown(
    options=['All'] + sorted(store_view['segment'].unique().tolist()),
    value='All',
    description='Segment:'
)

def plot_store_perf(segment):
    if segment == 'All':
        df_plot = store_view.copy()
        title = 'Store performance - all segments'
    else:
        df_plot = store_view[store_view['segment'] == segment]
        title = f'Store performance - {segment} segment'

    fig = px.bar(
        df_plot,
        x='Store_ID',
        y='sales',
        color='segment',
        hover_data=['Discount_Percentage', 'Stock_On_Hand', 'Store_Rating'],
        title=title
    )
    fig.update_layout(xaxis_title='Store', yaxis_title='Avg sales')
    fig.show()

widgets.interact(plot_store_perf, segment=segment_dropdown);

In [ ]:
# Time series + forecast view
# dropdown for store-level time-series (aggregated from original data)
store_dropdown = widgets.Dropdown(
    options=['All'] + sorted(data['Store_ID'].unique().tolist()),
    value='All',
    description='Store:'
)

def plot_time_series(store_id):
    if store_id == 'All':
        ts = data.groupby('date', as_index=False)['sales'].sum()
        title = 'Total sales over time - all stores'
    else:
        ts = data[data['Store_ID'] == store_id] \
                .groupby('date', as_index=False)['sales'].sum()
        title = f'Sales over time - {store_id}'

    fig = px.line(ts, x='date', y='sales', title=title)
    fig.update_layout(xaxis_title='Date', yaxis_title='Sales')
    fig.show()

widgets.interact(plot_time_series, store_id=store_dropdown);

In [ ]:
# Building df with just date and sales for time‑series plotting
df = data[['date', 'sales']].copy()
df = df.sort_values('date').reset_index(drop=True)

In [ ]:
#  Looking at how recent sales compare with the model forecast
#    (this is useful for managers to see near-term trends)

lookback_days = 30  # last 30 days of actuals
recent_actuals = df.tail(lookback_days).copy()

fig = px.line(
    recent_actuals,
    x="date",
    y="sales",
    title=f"Recent sales vs next 6‑day forecast (last {lookback_days} days)"
)

fig.add_scatter(
    x=future_df["date"],
    y=future_df["forecast_sales"],
    mode="lines+markers",
    name="Forecast",
)

fig.update_layout(
    xaxis_title="Date",
    yaxis_title="Sales",
    legend_title="Series"
)

fig.show()


In [ ]:
# Comparing linear time‑lag model vs random forest feature model

print("Linear lag model (test set):")
print(f"RMSE: {rmse:.2f}")
print(f"R²  : {r2:.4f}")

print("\nRandom Forest feature model (test set):")
print(f"RMSE: {rmse_rf:.2f}")
print(f"R²  : {r2_rf:.4f}")


In [ ]:
import pandas as pd

model_compare = pd.DataFrame({
    "model": ["Linear_lags", "RandomForest_features"],
    "rmse": [rmse, rmse_rf],
    "r2": [r2, r2_rf],
})
model_compare


In [ ]:
def full_forecast_pipeline(n_future=6, freq="D"):
    """
    Run the lag‑based model pipeline and return forecast dataframe.
    """
    # using existing df, model, and forecast_future
    last_12 = df["sales"].iloc[-12:].values
    future_vals = forecast_future(model, last_12, n_steps=n_future)

    last_date = df["date"].iloc[-1]
    future_dates = pd.date_range(
        start=last_date + pd.Timedelta(1, unit=freq[0]),
        periods=n_future,
        freq=freq
    )

    return pd.DataFrame({
        "date": future_dates,
        "forecast_sales": future_vals
    })

future_df_2 = full_forecast_pipeline(n_future=6, freq="D")
future_df_2.head()


In [ ]:
def forecast_store(store_id, n_future=6, freq="D"):
    """
    Build a simple per‑store time‑series and forecast next n_future days
    using the global lag model.
    """
    store_ts = data[data["Store_ID"] == store_id].copy()
    store_ts = store_ts.sort_values("date")
    df_store = store_ts[["date", "sales"]].copy()
    df_store["t"] = np.arange(1, len(df_store) + 1)

    last_12 = df_store["sales"].iloc[-12:].values
    future_vals = forecast_future(model, last_12, n_steps=n_future)

    last_date = df_store["date"].iloc[-1]
    future_dates = pd.date_range(
        start=last_date + pd.Timedelta(1, unit=freq[0]),
        periods=n_future,
        freq=freq
    )

    return pd.DataFrame({
        "Store_ID": store_id,
        "date": future_dates,
        "forecast_sales": future_vals
    })

example_store_forecast = forecast_store("STR_101", n_future=6)
example_store_forecast


In [ ]:
# Exporting per‑store performance and behavior
store_perf.to_csv("store_performance_segments.csv", index=False)
store_behavior.to_csv("store_behavior_summary.csv", index=False)
store_view.to_csv("store_view_for_dashboard.csv", index=False)

# Exporting model comparison
model_compare.to_csv("model_comparison.csv", index=False)

# Exporting latest global forecast
future_df_2.to_csv("global_6day_forecast.csv", index=False)
example_store_forecast.to_csv("STR_101_6day_forecast.csv", index=False)


In [ ]:
def run_full_sales_forecasting_pipeline(store_id="STR_101",
                                        n_future=6,
                                        freq="D"):
    """
    Run main steps and return a dictionary of key outputs.
    """
    outputs = {}
    outputs["model_compare"] = model_compare.copy()
    outputs["global_forecast"] = full_forecast_pipeline(
        n_future=n_future, freq=freq
    )
    outputs["store_forecast"] = forecast_store(
        store_id=store_id, n_future=n_future, freq=freq
    )
    outputs["store_perf"] = store_perf.copy()
    outputs["store_behavior"] = store_behavior.copy()
    outputs["store_view"] = store_view.copy()
    return outputs

results = run_full_sales_forecasting_pipeline(store_id="STR_101",
                                              n_future=6,
                                              freq="D")
results["global_forecast"].head()


In [ ]:
def summarize_results():
    best_model = model_compare.sort_values("rmse").iloc[0]
    print(f"Best model: {best_model['model']} "
          f"(RMSE={best_model['rmse']:.0f}, R²={best_model['r2']:.3f})")
    print("\nTop 3 stores by average sales:")
    print(store_sales.head(3))
    print("\nTop 3 product categories by average sales:")
    print(cat_sales.head(3))

summarize_results()


In [ ]:
#Quick seasonality view (monthly)
monthly_sales = (
    data
    .set_index("date")["sales"]
    .resample("M")
    .sum()
    .reset_index()
)

fig = px.line(
    monthly_sales,
    x="date",
    y="sales",
    title="Monthly sales seasonality (all stores)"
)
fig.update_layout(xaxis_title="Month", yaxis_title="Total sales")
fig.show()

In [ ]:
# Compact manager summary table
manager_summary = pd.DataFrame({
    "metric": [
        "Best model",
        "Best model RMSE",
        "Best model R²",
        "Top store by avg sales",
        "Top category by avg sales"
    ],
    "value": [
        model_compare.sort_values("rmse").iloc[0]["model"],
        f"{model_compare.sort_values('rmse').iloc[0]['rmse']:.0f}",
        f"{model_compare.sort_values('rmse').iloc[0]['r2']:.3f}",
        store_sales.index[0],
        cat_sales.index[0]
    ]
})
manager_summary


In [ ]:
# Interactive per‑store 6‑day forecast viewer
store_forecast_dropdown = widgets.Dropdown(
    options=sorted(data["Store_ID"].unique().tolist()),
    value="STR_101",
    description="Store:"
)

def show_store_forecast(store_id):
    df_fc = forecast_store(store_id, n_future=6, freq="D")
    display(df_fc)

widgets.interact(show_store_forecast, store_id=store_forecast_dropdown);


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# 1) Using store_view and rename existing cols to clearer names
store_features = store_view.copy()
store_features = store_features.rename(columns={
    "sales": "avg_sales"
})

# 2) enabling feature columns that really exist
feature_cols = ["avg_sales", "Discount_Percentage", "Stock_On_Hand", "Store_Rating"]

X = store_features[feature_cols].copy()

# 3) Scale and cluster
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

kmeans = KMeans(n_clusters=4, random_state=42, n_init="auto")
store_features["segment_kmeans"] = kmeans.fit_predict(X_scaled)

# 4)  map segment labels to friendly names
segment_names = {
    0: "Cluster 0",
    1: "Cluster 1",
    2: "Cluster 2",
    3: "Cluster 3",
}
store_features["segment_name_kmeans"] = store_features["segment_kmeans"].map(segment_names)

store_features.head()


In [ ]:
# Cluster-level summary: understanding each segment
cluster_profile = (
    store_features
    .groupby("segment_kmeans")
    [["avg_sales", "Discount_Percentage", "Stock_On_Hand", "Store_Rating"]]
    .mean()
    .round(2)
)

cluster_profile


In [ ]:
import plotly.express as px
# Visualizing store segments using sales, ratings, discounts, and stock with an interactive scatter plot


fig = px.scatter(
    store_features,
    x="avg_sales",
    y="Store_Rating",
    color="segment_kmeans",
    hover_data=["Store_ID", "Discount_Percentage", "Stock_On_Hand"],
    title="Store segmentation based on sales and rating"
)
fig.update_layout(xaxis_title="Avg sales", yaxis_title="Store rating")
fig.show()


In [ ]:
def show_management_dashboard(selected_store="STR_101"):
    display(manager_summary.style.set_caption("Executive summary"))

# Management dashboard: show executive summary, top 10 segmented stores, and 6-day forecast for a selected store

    print("\nTop 10 stores with segment labels:")
    cols_to_show = [
        "Store_ID",
        "avg_sales",
        "Discount_Percentage",
        "Stock_On_Hand",
        "Store_Rating",
        "segment",
        "segment_kmeans"
    ]
    display(
        store_features[cols_to_show]
        .sort_values("avg_sales", ascending=False)
        .head(10)
    )

    print(f"\nNext 6-day forecast for {selected_store}:")
    display(forecast_store(selected_store, n_future=6, freq="D"))

show_management_dashboard("STR_101")


In [ ]:
# Creating folder in Colab/Drive if needed
import os

output_dir = "retail_sales_outputs"
os.makedirs(output_dir, exist_ok=True)

# Exporting key tables
data.to_csv(os.path.join(output_dir, "retail_sales_cleaned.csv"), index=False)
store_view.to_csv(os.path.join(output_dir, "store_view_with_segments.csv"), index=False)
store_features.to_csv(os.path.join(output_dir, "store_features_kmeans.csv"), index=False)
model_compare.to_csv(os.path.join(output_dir, "model_comparison.csv"), index=False)
manager_summary.to_csv(os.path.join(output_dir, "manager_summary.csv"), index=False)


In [ ]:
# Global multi-day forecast
global_forecast = full_forecast_pipeline(n_future=30, freq="D")
global_forecast.to_csv(
    os.path.join(output_dir, "global_30day_forecast.csv"),
    index=False
)

#  exporting forecasts for all stores
all_store_forecasts = []

for s in sorted(data["Store_ID"].unique()):
    fc = forecast_store(s, n_future=7, freq="D")
    all_store_forecasts.append(fc)

all_store_forecasts = pd.concat(all_store_forecasts, ignore_index=True)
all_store_forecasts.to_csv(
    os.path.join(output_dir, "all_stores_7day_forecast.csv"),
    index=False
)


In [ ]:
def run_full_project(store_id="STR_101"):
  # Wrapper function to run the full project pipeline and collect key outputs for reporting and dashboards

    """
    Helper to run the key pieces of the project
    and return objects useful for reports/dashboards.
    """
    outputs = {}
    outputs["manager_summary"] = manager_summary
    outputs["cluster_profile"] = cluster_profile
    outputs["store_features"] = store_features
    outputs["global_forecast_30d"] = full_forecast_pipeline(n_future=30, freq="D")
    outputs["store_forecast_7d"] = forecast_store(store_id, n_future=7, freq="D")
    return outputs

project_outputs = run_full_project("STR_101")
project_outputs["manager_summary"]


In [ ]:
# Quick KPI-style metrics
total_revenue = data["sales"].sum()
avg_daily_revenue = (
    data.groupby("date")["sales"].sum().mean()
)

kpi_df = pd.DataFrame({
    "KPI": [
        "Total revenue (period)",
        "Avg daily revenue",
        "Number of stores",
        "Number of product categories"
    ],
    "Value": [
        f"{total_revenue:,.0f}",
        f"{avg_daily_revenue:,.0f}",
        data["Store_ID"].nunique(),
        data["Product_Category"].nunique()
    ]
})
kpi_df


In [ ]:
# Category-wise monthly trend
cat_monthly = (
    data
    .set_index("date")
    .groupby("Product_Category")["sales"]
    .resample("M")
    .sum()
    .reset_index()
)

fig = px.line(
    cat_monthly,
    x="date",
    y="sales",
    color="Product_Category",
    title="Monthly sales by product category"
)
fig.update_layout(xaxis_title="Month", yaxis_title="Sales")
fig.show()


In [ ]:
# Example: saving a matplotlib figure
import matplotlib.pyplot as plt

daily = data.groupby("date")["sales"].sum().reset_index()
plt.figure(figsize=(10,4))
plt.plot(daily["date"], daily["sales"])
plt.title("Daily total sales")
plt.xlabel("Date")
plt.ylabel("Sales")
plt.tight_layout()
plt.savefig(os.path.join(output_dir, "daily_total_sales.png"))
plt.close()


In [ ]:
import matplotlib.pyplot as plt

# Daily total sales line chart and save to file
daily = data.groupby("date")["sales"].sum().reset_index()

plt.figure(figsize=(10, 4))
plt.plot(daily["date"], daily["sales"])
plt.title("Daily total sales")
plt.xlabel("Date")
plt.ylabel("Sales")
plt.xticks(rotation=45)
plt.tight_layout()

plt.savefig(os.path.join(output_dir, "daily_total_sales.png"))
plt.close()


In [ ]:
print("Files created in", output_dir)
for f in sorted(os.listdir(output_dir)):
    print("-", f)


In [ ]:
import ipywidgets as widgets
from IPython.display import display

def full_dashboard():
    # Dropdowns
    store_dropdown = widgets.Dropdown(
        options=["All"] + sorted(data["Store_ID"].unique().tolist()),
        value="All",
        description="Store:",
        layout=widgets.Layout(width="300px")
    )

    seg_dropdown = widgets.Dropdown(
        options=["All", "Low", "Medium", "High"],
        value="All",
        description="Segment:",
        layout=widgets.Layout(width="300px")
    )

    out = widgets.Output()

    def refresh_dashboard(change=None):
        with out:
            out.clear_output()

            # 1) KPIs and manager summary
            print("Executive summary")
            display(manager_summary)

            # 2) Segment view
            print("\nStore segment overview (top 10 by avg sales):")
            cols = [
                "Store_ID",
                "avg_sales",
                "Discount_Percentage",
                "Stock_On_Hand",
                "Store_Rating",
                "segment",
                "segment_kmeans"
            ]
            df_seg = store_features[cols].copy()
            if seg_dropdown.value != "All":
                df_seg = df_seg[df_seg["segment"] == seg_dropdown.value]
            display(df_seg.sort_values("avg_sales", ascending=False).head(10))

            # 3) Time series + forecast for selected store
            selected_store = store_dropdown.value
            print(f"\nTime series and forecast for: {selected_store}")
            plot_time_series(selected_store)  # uses your existing function
            print("\nNext 7-day forecast table:")
            if selected_store == "All":
                display(global_forecast.head(7))
            else:
                display(forecast_store(selected_store, n_future=7, freq="D"))

    store_dropdown.observe(refresh_dashboard, names="value")
    seg_dropdown.observe(refresh_dashboard, names="value")

    controls = widgets.HBox([store_dropdown, seg_dropdown])
    display(controls, out)
    refresh_dashboard()

full_dashboard()


In [ ]:
plt.savefig('demo_plot.png', dpi=300, bbox_inches='tight')

In [ ]:
#In this project, I worked as a data scientist for a retail chain to build an end-to-end sales analytics and forecasting system. Using the retail sales dataset (Jan 2023–Dec 2024), I performed data cleaning, feature engineering, and exploratory analysis across stores and product categories to understand trends, seasonality, and key revenue drivers. I then developed forecasting models (time-series style with lags and a tree-based ML model), evaluated them with RMSE and R², and generated short-term forecasts at both overall and per-store levels. I also created store segments (rule-based and KMeans) based on sales, discounts, stock, and ratings to distinguish high-value, stable, and volatile stores. Finally, I built interactive in-notebook dashboards that show KPIs, store segments, time-series plots, and forecast tables so management can explore insights and use the model outputs for inventory and planning decisions.​​